In [10]:
import dynamiqs as dq
import jax.numpy as jnp
import jax
from time import time

jax.config.update("jax_enable_x64", True)
dq.set_precision("double")
dq.set_device("cpu")

n = 2**6
N = 1
batch_size = 2**4
rk4_steps = 100
use_rk4 = True

a = dq.destroy(n)
H = a.dag() @ a
jump_ops = [a]
psi0 = dq.coherent(n, 1.0)
rho = psi0 @ psi0.dag()
rho = dq.to_jax(rho)

def __lindblad_rhs(t, rho, H, Ls):
    comm = H @ rho - rho @ H
    drho = -1j * comm
    for L in Ls:
        Lrho = L @ rho
        drho += Lrho @ L.conj().T \
                - 0.5 * (L.conj().T @ L @ rho + rho @ L.conj().T @ L)
    return drho

def __rk4_step(f, y, t, dt, *args):
    """One RK4 step for y' = f(t, y, *args)."""
    k1 = f(t,         y,             *args)
    k2 = f(t + dt/2., y + dt/2.*k1,  *args)
    k3 = f(t + dt/2., y + dt/2.*k2,  *args)
    k4 = f(t + dt,    y + dt*k3,     *args)
    return y + dt/6.*(k1 + 2*k2 + 2*k3 + k4)

def lindblad_rk4(rho0, ts, H, Ls):
    def f(t, rho, H, Ls):
        return __lindblad_rhs(t, rho, H, Ls)

    rho = rho0
    for i in range(len(ts)-1):
        dt = ts[i+1] - ts[i]
        rho = __rk4_step(f, rho, ts[i], dt, H, Ls)
    return rho

def solve1(rho):
    result = dq.mesolve(
        H,
        jump_ops,
        rho,
        tsave=[1],
        options=dq.Options(
            t0=0,
            progress_meter=False
        )
    )
    return jnp.real(result.states[-1].to_jax()).sum()/N

def solve2(rho):
    rho_final = lindblad_rk4(rho, jnp.linspace(0, 1, rk4_steps), H, jump_ops)
    return jnp.real(rho_final).sum()/N

if use_rk4:
    solve = solve2
    H = H.to_jax()
    jump_ops = [L.to_jax() for L in jump_ops]
else:
    solve = solve1

def f(rho_batch):
    def f2(rho):
        sum = 0.0
        for _ in range(N):
            sum = sum + solve(rho)
        return sum

    return jax.vmap(f2)(rho_batch).sum()

f1 = jax.jit(f)
f2 = jax.jit(jax.value_and_grad(f))

rho_batch = jnp.stack([rho] * batch_size)

f1(rho_batch)  # Warm-up JIT
f2(rho_batch)  # Warm-up JIT

start = time()
for _ in range(10):
    val = f1(rho_batch)
print("time", (time() - start))

start = time()
for _ in range(10):
    val, grad = f2(rho_batch)
print("time with grad", (time() - start))


time 0.003354787826538086
time with grad 0.0015530586242675781
